
# Árboles y ensambles, Notebook 4
## Ensambles: bagging, random forest, AdaBoost y gradient boosting

**Preparado por:** David Díaz, con asistencia de Claude (Anthropic) · **Entorno:** Google Colab

### De qué se trata

Un árbol solo es inestable: cambia unas filas y cambia la raíz (lo vimos en el Notebook 3).
Los **ensambles** usan eso a favor: en vez de un árbol, muchos árboles imperfectos que votan
o que se suman. Hay dos familias, y conviene tenerlas separadas desde el principio:

| Familia | Cómo entrena | Qué reduce | Ejemplos |
|---|---|---|---|
| **Bagging** | Muchos árboles **en paralelo**, cada uno con una copia sorteada de los datos; al final votan | La **varianza** (la inestabilidad) | Bagging (Breiman, 1996), Random forest (Breiman, 2001) |
| **Boosting** | Muchos árboles chicos **en serie**, cada uno concentrado en lo que los anteriores hicieron mal; al final se suman | El **sesgo** (lo que un árbol chico no alcanza a captar) | AdaBoost (Freund y Schapire, 1996), Gradient boosting (Friedman, 2001), XGBoost, LightGBM, CatBoost |

Hoy construimos cada uno **a mano** con las 219 empresas, para ver el mecanismo, y después los
soltamos con `scikit-learn` sobre 30.000 clientes de tarjetas de crédito, donde se ve para qué
sirven.

### Qué vas a aprender hoy

1. Bagging: bootstrap, votación y el error "fuera de la bolsa".
2. Random forest: el segundo sorteo, y la importancia de variables.
3. AdaBoost: reponderar las empresas difíciles, ronda a ronda.
4. Gradient boosting: sumar escaleras que corrigen el residuo, y la familia XGBoost / LightGBM / CatBoost.
5. Cuándo un ensamble le gana a un árbol, y cuándo no.


In [1]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings, time

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from scipy.stats import binom

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
print("Listo.")


Listo.



## 1. Los datos, otra vez

Las mismas 219 empresas y la misma partición de los Notebooks 2 y 3. El punto de referencia es
el árbol de profundidad 2: 76,7% en entrenamiento y 82,2% en prueba.


| Ratio | Qué mide |
|---|---|
| `deuda_activos` | deuda total / activos totales (endeudamiento) |
| `razon_corriente` | activo circulante / pasivo circulante (liquidez) |
| `ventas_deuda` | ventas / deuda total (capacidad de servir la deuda) |
| `ln_activos` | logaritmo de los activos totales (tamaño) |
| `roa` | utilidad neta / activos totales (rentabilidad) |
| `impago` | **lo que queremos predecir:** 1 si la empresa cayó en impago, 0 si no |


In [2]:

# Las 219 empresas con ratios financieros (la tabla de impago del curso, con 5 de sus ratios).
# Está pegada aquí mismo para que el notebook no dependa de ningún archivo externo.
from io import StringIO
IMPAGO_CSV = """deuda_activos,razon_corriente,ventas_deuda,ln_activos,roa,impago
0.374,2.337,1.436,14.697,0.073,0.0
0.435,2.638,1.385,14.877,0.042,0.0
0.443,2.099,0.452,14.931,0.016,0.0
0.23,2.506,4.217,15.586,-0.041,0.0
0.317,2.273,3.116,15.708,0.021,0.0
0.312,3.282,4.842,15.785,0.047,0.0
0.628,1.386,2.91,14.066,0.064,0.0
0.64,1.777,2.895,14.236,0.068,0.0
0.719,1.44,1.505,14.697,0.075,0.0
0.551,1.078,1.944,15.564,0.113,0.0
0.541,0.991,2.091,15.581,0.022,0.0
0.575,0.995,1.787,15.738,0.035,0.0
0.454,3.266,7.321,14.877,0.064,0.0
0.574,3.017,3.159,14.457,0.07,0.0
0.526,3.333,1.962,14.399,0.077,0.0
0.724,0.8,1.625,14.036,0.016,0.0
0.508,1.406,4.356,14.349,0.406,0.0
0.494,1.051,2.299,14.515,0.117,0.0
0.572,1.557,2.106,14.016,0.126,0.0
0.578,1.498,2.033,14.293,0.086,0.0
0.558,1.605,2.035,14.52,0.099,0.0
0.314,2.127,10.633,13.738,0.087,0.0
0.467,1.661,5.599,14.219,0.094,0.0
0.499,1.482,3.109,14.424,0.067,0.0
0.259,3.734,5.692,14.172,0.234,0.0
0.202,4.804,7.312,14.417,0.195,0.0
0.268,3.663,6.08,14.897,0.228,0.0
0.409,1.671,4.523,14.253,0.12,0.0
0.307,2.171,6.817,14.38,0.172,0.0
0.258,3.481,10.557,14.454,0.274,0.0
0.393,1.55,3.81,14.68,0.024,0.0
0.558,1.102,2.597,14.108,0.005,0.0
0.432,1.365,3.534,14.01,0.119,0.0
0.53,1.281,3.099,12.687,0.272,0.0
0.244,3.411,16.602,13.143,0.489,0.0
0.354,2.877,3.112,13.311,0.075,0.0
0.381,1.56,2.749,14.696,0.074,0.0
0.411,2.198,2.46,14.839,0.043,0.0
0.401,1.954,2.067,14.915,0.037,0.0
0.387,1.955,2.616,14.835,0.342,0.0
0.438,1.029,2.619,14.929,0.023,0.0
0.413,1.44,2.869,14.889,-0.011,0.0
0.891,1.118,2.023,13.415,0.036,0.0
1.052,0.741,1.8,14.355,-0.093,0.0
0.466,0.899,1.509,14.364,0.034,0.0
0.491,1.196,2.335,14.414,-0.017,0.0
0.505,1.053,2.678,14.631,0.076,0.0
0.354,2.45,3.08,14.453,0.084,0.0
0.464,1.843,2.521,14.559,0.119,0.0
0.566,1.286,1.675,14.838,0.111,0.0
0.543,1.766,2.619,14.933,0.029,0.0
0.855,1.131,1.805,15.005,0.029,0.0
0.558,1.744,2.516,15.074,0.028,0.0
0.736,1.153,1.564,14.292,0.054,0.0
0.708,1.192,1.352,14.432,0.034,0.0
0.657,1.269,1.395,14.387,0.027,0.0
0.653,1.061,1.588,14.303,0.04,0.0
0.619,1.005,2.113,14.235,0.038,0.0
0.197,9.632,5.15,14.586,0.032,0.0
0.284,1.461,6.585,13.479,0.11,0.0
0.3,1.438,5.251,13.541,0.012,0.0
0.283,2.928,5.778,13.561,0.016,0.0
0.476,1.788,5.329,13.143,0.123,0.0
0.47,1.759,5.033,13.329,0.13,0.0
0.459,2.104,4.49,13.372,0.076,0.0
0.388,2.409,5.319,13.283,0.201,0.0
0.547,1.979,2.09,13.806,0.15,0.0
0.568,1.812,1.937,13.914,0.071,0.0
0.461,1.923,1.499,14.059,-0.013,0.0
0.538,2.174,1.801,14.208,0.025,0.0
0.497,1.878,1.917,14.284,0.075,0.0
0.208,3.306,5.181,15.336,-0.003,0.0
0.241,3.005,4.709,15.428,0.011,0.0
0.222,3.11,3.776,15.427,0.021,0.0
0.392,2.268,6.865,12.39,0.334,0.0
0.252,5.229,7.751,12.318,0.176,0.0
0.849,1.56,1.326,12.304,-0.186,0.0
0.36,2.671,5.196,12.142,0.186,0.0
0.028,34.514,71.252,12.295,0.292,0.0
0.631,1.369,1.835,13.284,0.136,0.0
0.507,1.464,1.625,13.732,-0.217,0.0
0.578,1.577,2.553,13.963,0.02,0.0
0.448,1.681,0.566,13.768,0.024,0.0
0.363,2.721,2.762,13.301,0.446,0.0
0.403,2.199,2.673,13.506,0.067,0.0
0.4,2.237,1.139,13.576,0.043,0.0
0.432,1.579,2.39,14.7,0.066,0.0
0.464,1.763,2.067,14.843,0.059,0.0
0.475,1.707,1.833,14.95,0.06,0.0
0.089,8.849,16.344,15.39,0.013,0.0
0.125,7.175,11.171,15.513,0.059,0.0
0.125,7.156,12.795,15.55,0.041,0.0
0.539,1.764,3.733,15.384,0.088,0.0
0.502,1.79,3.797,15.397,0.035,0.0
0.522,1.572,2.624,15.391,0.029,0.0
0.77,0.383,1.984,15.834,-0.046,0.0
0.746,0.465,3.31,16.041,0.057,0.0
0.666,0.539,3.919,16.051,0.018,0.0
0.14,8.346,5.975,15.361,0.008,0.0
0.183,5.927,4.699,15.443,0.026,0.0
0.24,4.46,3.828,15.549,0.016,0.0
0.749,1.39,2.739,15.184,0.015,0.0
0.72,1.212,2.976,15.14,0.006,0.0
0.824,0.937,3.12,15.016,-0.089,0.0
0.127,0.463,3.018,13.706,0.036,0.0
0.584,3.254,1.629,14.581,0.053,0.0
0.565,0.66,1.621,14.753,0.085,0.0
0.853,3.41,0.45,13.939,0.012,0.0
0.836,4.713,0.337,14.072,0.007,0.0
0.968,2.708,0.249,14.346,-0.016,0.0
0.387,2.077,1.735,15.354,-0.039,0.0
0.398,2.395,2.933,15.532,0.059,0.0
0.327,2.428,1.813,15.518,0.036,0.0
0.635,1.999,0.061,15.292,-0.005,0.0
0.279,4.784,0.637,14.664,0.016,0.0
0.162,2.572,1.5,14.58,0.057,0.0
0.21,1.923,15.689,12.954,0.112,0.0
0.434,0.844,4.415,12.837,-0.246,0.0
0.271,1.323,5.378,12.849,0.17,0.0
0.416,2.122,5.392,13.508,0.098,0.0
0.409,2.225,4.438,13.477,0.018,0.0
0.262,3.185,5.914,13.392,0.128,0.0
0.23,4.172,5.796,14.956,0.218,0.0
0.085,11.333,15.115,15.14,0.246,0.0
0.134,6.66,9.398,15.292,0.101,0.0
0.759,2.52,2.074,13.84,0.084,0.0
0.77,1.92,1.572,14.162,0.031,0.0
0.675,2.107,1.621,13.957,0.036,0.0
0.601,2.387,1.541,13.442,-0.214,0.0
0.733,1.376,1.151,13.28,-0.11,0.0
0.822,1.461,1.286,13.478,-0.026,0.0
0.15,9.243,7.907,13.92,0.068,0.0
0.103,13.87,11.417,14.067,0.135,0.0
0.054,35.477,17.746,14.134,0.107,0.0
0.672,1.436,1.373,13.419,0.145,1.0
0.292,0.205,1.514,13.547,0.171,1.0
0.614,1.341,1.041,13.703,0.069,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,0.148,14.649,0.092,1.0
0.616,2.062,1.508,14.571,0.074,1.0
0.622,1.136,0.098,14.945,0.043,1.0
0.731,1.134,0.852,15.429,0.041,1.0
0.747,0.97,0.859,15.636,0.037,1.0
0.485,1.984,3.443,13.503,0.059,1.0
0.599,1.421,2.111,13.788,0.033,1.0
0.685,1.229,1.801,13.965,0.009,1.0
0.675,0.869,1.083,13.874,0.096,1.0
0.828,0.878,0.992,14.654,0.018,1.0
0.696,1.079,3.177,14.413,0.072,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.743,0.921,1.933,13.546,0.185,1.0
0.752,1.488,2.063,13.671,0.202,1.0
1.074,1.367,1.418,13.605,0.13,1.0
0.297,0.154,0.94,16.088,0.064,1.0
0.268,1.354,0.861,16.111,0.025,1.0
0.09,1.484,2.472,16.149,0.019,1.0
0.405,3.01,2.249,15.232,0.022,1.0
0.488,2.13,2.073,15.416,0.03,1.0
0.528,1.87,2.078,15.514,0.071,1.0
0.715,1.106,1.3,15.805,0.055,1.0
0.66,1.287,1.822,15.719,0.018,1.0
0.678,1.252,1.754,15.89,0.029,1.0
0.631,1.578,1.041,14.786,0.02,1.0
0.635,1.733,1.055,14.82,0.026,1.0
0.659,1.585,1.171,14.771,0.028,1.0
0.689,1.368,0.95,12.36,-0.033,1.0
0.464,1.238,1.069,12.662,-0.162,1.0
0.845,1.356,0.552,12.717,-0.352,1.0
0.548,1.741,0.697,16.077,0.002,1.0
0.595,1.234,0.758,16.058,0.0,1.0
0.768,0.919,0.355,16.536,-0.08,1.0
0.535,3.214,3.409,12.725,0.186,1.0
0.588,2.784,4.476,12.623,0.303,1.0
0.683,2.317,4.213,12.453,0.327,1.0
0.42,1.553,1.755,15.717,0.012,1.0
0.397,2.599,2.095,15.712,0.028,1.0
0.348,2.223,1.268,15.703,0.021,1.0
0.592,1.518,2.17,14.555,0.047,1.0
0.608,1.401,1.509,14.678,0.014,1.0
0.661,0.001,0.941,14.767,-0.013,1.0
1.48,0.225,0.001,12.488,-0.078,1.0
1.575,0.324,0.044,12.488,-0.009,1.0
0.638,0.87,1.662,15.163,-0.005,1.0
0.624,0.906,1.764,15.229,0.03,1.0
0.631,1.037,1.033,8.626,0.086,1.0
0.27,2.783,10.977,13.105,0.19,1.0
0.471,2.03,6.381,13.449,0.224,1.0
0.292,3.383,9.848,13.567,0.22,1.0
0.701,1.253,3.098,15.845,0.056,1.0
0.799,0.047,2.534,15.942,0.048,1.0
0.772,1.24,2.471,16.078,0.054,1.0
1.181,0.804,0.944,12.801,0.075,1.0
0.464,1.293,2.927,12.743,0.092,1.0
0.514,1.105,2.764,12.839,0.082,1.0
0.386,2.301,6.078,12.668,0.399,1.0
0.723,1.327,1.854,13.612,0.125,1.0
0.695,1.38,1.813,13.589,0.085,1.0
0.638,1.322,4.07,13.505,0.286,1.0
0.555,1.614,3.204,13.949,0.13,1.0
0.594,1.528,2.718,14.176,0.125,1.0
1.401,0.709,0.208,12.076,-0.146,1.0
0.916,0.616,1.27,12.629,0.316,1.0
0.75,0.489,2.46,12.725,0.29,1.0
0.445,9.164,0.0,11.167,-0.737,1.0
0.767,0.664,1.149,12.778,-0.779,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.775,1.025,3.294,13.079,0.112,1.0
0.731,0.932,4.179,12.889,0.126,1.0
0.842,0.673,1.911,13.274,-0.04,1.0
0.733,1.226,2.588,13.42,0.091,1.0
0.707,1.325,2.385,13.461,0.06,1.0
1.209,0.656,1.554,13.272,-0.565,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,1.479,14.649,0.092,1.0
0.616,2.062,2.277,14.571,0.074,1.0"""
impago = pd.read_csv(StringIO(IMPAGO_CSV))
print("Empresas:", len(impago), "  Fracción en impago:", round(impago["impago"].mean(), 3))
impago.describe().round(3).T[["mean", "min", "50%", "max"]]


Empresas: 219   Fracción en impago: 0.388


,mean,min,50%,max
deuda_activos,0.560,0.028,0.539,1.732
razon_corriente,2.380,0.001,1.560,35.477
ventas_deuda,3.474,0.000,2.170,71.252
ln_activos,14.199,8.626,14.303,16.536
roa,0.057,-0.779,0.055,0.489
impago,0.388,0.000,0.000,1.000


In [3]:

# La misma partición en todos los notebooks del set: 146 empresas para entrenar, 73 para probar
RATIOS = ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"]
rng = np.random.default_rng(2)
orden = rng.permutation(len(impago))
es_prueba = np.zeros(len(impago), dtype=bool); es_prueba[orden[:73]] = True
impago["conjunto"] = np.where(es_prueba, "prueba", "entrenamiento")

X = impago[RATIOS]; y = impago["impago"]
X_train, y_train = X[~es_prueba], y[~es_prueba]
X_test, y_test = X[es_prueba], y[es_prueba]
print("Entrenamiento:", len(X_train), "  Prueba:", len(X_test))
print("Fracción en impago: entrenamiento", round(y_train.mean(), 3), " prueba", round(y_test.mean(), 3))


Entrenamiento: 146   Prueba: 73
Fracción en impago: entrenamiento 0.397  prueba 0.37


In [4]:

def evaluar(modelo, nombre):
    acc_tr = accuracy_score(y_train, modelo.predict(X_train)); acc_te = accuracy_score(y_test, modelo.predict(X_test))
    print(f"{nombre:44s} entrenamiento {acc_tr:.3f}   prueba {acc_te:.3f}")
    return acc_te

arbol2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
arbol_full = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
evaluar(arbol2, "árbol de profundidad 2 (referencia)")
evaluar(arbol_full, "árbol sin freno (28 hojas)");


árbol de profundidad 2 (referencia)          entrenamiento 0.767   prueba 0.822
árbol sin freno (28 hojas)                   entrenamiento 1.000   prueba 0.767



## 2. Bagging a mano: sortear, entrenar, votar

**Bootstrap** es sortear $n$ filas **con reemplazo** de una tabla de $n$ filas: algunas salen
dos o tres veces, otras ninguna. En promedio queda fuera un 37% ($1/e$) de las filas. Cada
sorteo da una tabla parecida pero distinta; un árbol entrenado en cada una da árboles parecidos
pero distintos; y al hacerlos votar, los errores que no comparten se cancelan.

**Por qué funciona.** La varianza del promedio de $B$ árboles correlacionados es

$$
\text{Var} = \rho \, \sigma^2 + \frac{1 - \rho}{B} \, \sigma^2
$$

donde $\sigma^2$ es la varianza de un árbol y $\rho$ la correlación entre dos árboles. Con $B$
grande queda $\rho \sigma^2$: promediar muchos árboles elimina la parte que no comparten, pero no
la que comparten. Bajar $\rho$ es lo que hace el random forest.


In [5]:

rng = np.random.default_rng(11)
B = 7
n = len(X_train)
muestras = [rng.integers(0, n, n) for _ in range(B)]       # índices sorteados con reemplazo

cuentas = pd.DataFrame({f"muestra {b+1}": np.bincount(m, minlength=n) for b, m in enumerate(muestras)})
print("Cuántas veces sale cada empresa en cada muestra (primeras 10 empresas):")
print(cuentas.head(10).T)
print()
print("Empresas que quedan fuera de cada muestra:", [(c == 0).sum() for _, c in cuentas.items()], "  (de 146; en promedio 1/e = 37%)")


Cuántas veces sale cada empresa en cada muestra (primeras 10 empresas):
           0  1  2  3  4  5  6  7  8  9
muestra 1  1  1  3  3  3  0  0  1  0  1
muestra 2  2  0  1  0  0  3  0  1  1  2
muestra 3  1  2  0  0  2  1  0  0  0  2
muestra 4  0  1  0  0  2  0  3  3  3  0
muestra 5  3  3  3  2  2  0  1  2  0  3
muestra 6  0  0  0  2  1  3  1  0  0  0
muestra 7  1  1  2  3  0  0  2  0  1  0

Empresas que quedan fuera de cada muestra: [np.int64(58), np.int64(58), np.int64(56), np.int64(50), np.int64(51), np.int64(57), np.int64(52)]   (de 146; en promedio 1/e = 37%)


In [6]:

arboles_bag = [DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train.iloc[m], y_train.iloc[m]) for m in muestras]
for b, a in enumerate(arboles_bag):
    print(f"árbol {b+1}: raíz {RATIOS[a.tree_.feature[0]]} ≤ {a.tree_.threshold[0]:.3f}   acierto prueba {accuracy_score(y_test, a.predict(X_test)):.3f}")

votos = np.column_stack([a.predict(X_test) for a in arboles_bag])
mayoria = (votos.mean(axis=1) > 0.5).astype(int)
print()
print(f"Árbol típico (promedio de los 7) en prueba: {np.mean([accuracy_score(y_test, a.predict(X_test)) for a in arboles_bag]):.3f}")
print(f"Votación de los 7 en prueba:                {accuracy_score(y_test, mayoria):.3f}")


árbol 1: raíz deuda_activos ≤ 0.583   acierto prueba 0.753
árbol 2: raíz ventas_deuda ≤ 1.326   acierto prueba 0.671
árbol 3: raíz deuda_activos ≤ 0.583   acierto prueba 0.699
árbol 4: raíz deuda_activos ≤ 0.622   acierto prueba 0.712
árbol 5: raíz razon_corriente ≤ 1.746   acierto prueba 0.712
árbol 6: raíz deuda_activos ≤ 0.588   acierto prueba 0.712
árbol 7: raíz ventas_deuda ≤ 1.313   acierto prueba 0.726

Árbol típico (promedio de los 7) en prueba: 0.712
Votación de los 7 en prueba:                0.753



Siete muestras, siete árboles con raíces distintas (ahí está la inestabilidad), y la votación
le gana al árbol típico del conjunto. Mira una empresa de prueba en detalle:


In [7]:

i = 0
print("Empresa de prueba 1:", X_test.iloc[i].round(3).to_dict(), "  real:", "impago" if y_test.iloc[i] else "paga")
print("Votos de los 7 árboles:", ["impago" if v else "paga" for v in votos[i]], "->", "impago" if mayoria[i] else "paga")


Empresa de prueba 1: {'deuda_activos': 0.374, 'razon_corriente': 2.337, 'ventas_deuda': 1.436, 'ln_activos': 14.697, 'roa': 0.073}   real: paga
Votos de los 7 árboles: ['paga', 'paga', 'paga', 'paga', 'paga', 'paga', 'paga'] -> paga



### El error fuera de la bolsa (OOB)

Bagging trae un regalo: cada árbol dejó fuera un 37% de las empresas, y esas empresas son
datos que **ese árbol no vio**. Para cada empresa de entrenamiento se hace votar solo a los
árboles que no la usaron, y se mide el acierto. Es una estimación de prueba gratis, sin gastar
las 73 empresas reservadas.


In [8]:

aciertos = []
for j in range(n):
    fuera = [b for b in range(B) if cuentas.iloc[j, b] == 0]
    if not fuera: continue
    voto = np.mean([arboles_bag[b].predict(X_train.iloc[[j]])[0] for b in fuera]) > 0.5
    aciertos.append(int(voto) == y_train.iloc[j])
print(f"Acierto fuera de la bolsa: {np.mean(aciertos):.3f} sobre {len(aciertos)} empresas   (acierto en prueba: {accuracy_score(y_test, mayoria):.3f})")


Acierto fuera de la bolsa: 0.736 sobre 140 empresas   (acierto en prueba: 0.753)



## 3. Random forest: el segundo sorteo

Si todos los árboles tienen la misma raíz, se parecen demasiado ($\rho$ alto) y votar ayuda
poco. Random forest agrega un segundo sorteo: en **cada corte**, cada árbol solo puede mirar
un subconjunto de las variables elegido al azar (`max_features`, típicamente la raíz cuadrada del
total). Así se obliga a los árboles a usar variables distintas y a parecerse menos.


In [9]:

bag100 = BaggingClassifier(DecisionTreeClassifier(random_state=0), n_estimators=100, oob_score=True, random_state=0).fit(X_train, y_train)
rf100 = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=0).fit(X_train, y_train)
rf100_d3 = RandomForestClassifier(n_estimators=100, max_depth=3, max_features=2, oob_score=True, random_state=0).fit(X_train, y_train)
evaluar(arbol2, "árbol de profundidad 2 (referencia)")
evaluar(bag100, f"bagging, 100 árboles (OOB {bag100.oob_score_:.3f})")
evaluar(rf100, f"random forest, 100 árboles (OOB {rf100.oob_score_:.3f})")
evaluar(rf100_d3, f"random forest, 100 árboles, prof. 3, 2 var. (OOB {rf100_d3.oob_score_:.3f})");


árbol de profundidad 2 (referencia)          entrenamiento 0.767   prueba 0.822
bagging, 100 árboles (OOB 0.733)             entrenamiento 1.000   prueba 0.795
random forest, 100 árboles (OOB 0.774)       entrenamiento 1.000   prueba 0.753
random forest, 100 árboles, prof. 3, 2 var. (OOB 0.753) entrenamiento 0.870   prueba 0.753



**Lectura honesta.** Con 219 empresas y 5 ratios, ni el bagging ni el random forest le ganan
al árbol de dos niveles en prueba. Un árbol chico bien elegido ya captura casi todo lo que hay
en estos datos, y con 73 empresas de prueba las diferencias son de una o dos empresas. Los
ensambles brillan con más datos, más variables y más ruido; lo vamos a ver en la sección 6.
Aquí lo que se ve es el mecanismo, no su mejor caso.

### Importancia de variables

Un bosque ya no se lee como un reglamento. A cambio entrega una **importancia** por variable:
cuánta impureza redujo cada una, sumada sobre todos los cortes de todos los árboles.


In [10]:

imp = pd.Series(rf100.feature_importances_, index=RATIOS).sort_values(ascending=False)
fig = px.bar(imp, orientation="h", title="Importancia de variables del random forest (reducción de Gini acumulada)", labels={"value": "importancia", "index": ""})
fig.show()
imp.round(3)


,0
ventas_deuda,0.254
ln_activos,0.243
deuda_activos,0.198
razon_corriente,0.178
roa,0.128



### Por qué votar funciona: el cálculo binomial

Si cada clasificador se equivoca con probabilidad $\varepsilon$ y los errores son
independientes, la mayoría de $n$ clasificadores se equivoca con probabilidad binomial:

$$
P(\text{mayoría se equivoca}) = \sum_{k > n/2} \binom{n}{k} \varepsilon^k (1 - \varepsilon)^{n - k}
$$

Con $\varepsilon = 0{,}35$ y 25 clasificadores, la mayoría falla el 6% de las veces. El supuesto
fuerte es la independencia: en la práctica los árboles se parecen ($\rho > 0$) y la mejora es
menor. Por eso importa el segundo sorteo.


In [11]:

filas = []
for eps in [0.2, 0.35, 0.45, 0.5, 0.6]:
    fila = {"ε (error de cada uno)": eps}
    for n_clf in [1, 5, 25, 101]:
        fila[f"{n_clf} clasificadores"] = binom.sf(n_clf // 2, n_clf, eps)   # P(X > n/2)
    filas.append(fila)
pd.DataFrame(filas).round(3)


,ε (error de cada uno),1 clasificadores,5 clasificadores,25 clasificadores,101 clasificadores
0,0.20,0.20,0.058,0.000,0.000
1,0.35,0.35,0.235,0.060,0.001
2,0.45,0.45,0.407,0.306,0.156
3,0.50,0.50,0.500,0.500,0.500
4,0.60,0.60,0.683,0.846,0.979



Con $\varepsilon < 0{,}5$ el error de la mayoría cae hacia cero al sumar clasificadores; con
$\varepsilon = 0{,}5$ votar no cambia nada; con $\varepsilon > 0{,}5$ votar **empeora**.

## 4. AdaBoost a mano: concentrarse en lo difícil

Boosting entrena en serie. El primer modelo mira todas las empresas por igual; el segundo da
más peso a las que el primero clasificó mal; y así. Al final votan con pesos: los modelos que
acertaron más pesan más. Los modelos base son **tocones**: árboles de un solo corte.

$$
\varepsilon_m = \sum_i w_i \, [h_m(x_i) \neq y_i]
\qquad
\alpha_m = \tfrac{1}{2} \ln \frac{1 - \varepsilon_m}{\varepsilon_m}
\qquad
w_i \leftarrow \frac{w_i \, e^{-\alpha_m y_i h_m(x_i)}}{Z}
\qquad
F(x) = \text{signo}\Big(\sum_m \alpha_m h_m(x)\Big)
$$

$w_i$ es el peso de la empresa $i$ (parten iguales, $1/n$); $h_m$ el tocón de la ronda $m$
($+1$ impago, $-1$ paga); $\varepsilon_m$ su error **ponderado**; $\alpha_m$ su peso en el voto
(grande si $\varepsilon$ es chico, cero si $\varepsilon = 0{,}5$); $Z$ normaliza los pesos para que
sumen 1. Las mal clasificadas ($y \cdot h = -1$) suben de peso; las bien clasificadas bajan.
Una propiedad bonita: después de reponderar, las empresas que el tocón recién elegido clasificó
mal pesan exactamente la mitad del total, así que ese tocón queda "neutralizado" y el siguiente
tiene que encontrar otra cosa.


In [12]:

y_pm = np.where(y_train == 1, 1, -1); y_pm_test = np.where(y_test == 1, 1, -1)
w = np.ones(n) / n
F_train = np.zeros(n); F_test = np.zeros(len(X_test))
for m in range(5):
    tocon = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_train, y_pm, sample_weight=w)
    h = tocon.predict(X_train); h_test = tocon.predict(X_test)
    eps = w[h != y_pm].sum()
    alfa = 0.5 * np.log((1 - eps) / eps)
    F_train += alfa * h; F_test += alfa * h_test
    w = w * np.exp(-alfa * y_pm * h); w = w / w.sum()
    print(f"ronda {m+1}: tocón {RATIOS[tocon.tree_.feature[0]]:16s} ≤ {tocon.tree_.threshold[0]:7.3f}   ε = {eps:.3f}   α = {alfa:.3f}   "
          f"peso de las mal clasificadas después: {w[h != y_pm].sum():.3f}   acumulado: entrenamiento {np.mean(np.sign(F_train) == y_pm):.3f}  prueba {np.mean(np.sign(F_test) == y_pm_test):.3f}")


ronda 1: tocón ventas_deuda     ≤   1.278   ε = 0.253   α = 0.540   peso de las mal clasificadas después: 0.500   acumulado: entrenamiento 0.747  prueba 0.685
ronda 2: tocón razon_corriente  ≤   1.534   ε = 0.296   α = 0.434   peso de las mal clasificadas después: 0.500   acumulado: entrenamiento 0.747  prueba 0.685
ronda 3: tocón ln_activos       ≤  13.718   ε = 0.325   α = 0.366   peso de las mal clasificadas después: 0.500   acumulado: entrenamiento 0.781  prueba 0.753
ronda 4: tocón razon_corriente  ≤   3.234   ε = 0.353   α = 0.302   peso de las mal clasificadas después: 0.500   acumulado: entrenamiento 0.795  prueba 0.740
ronda 5: tocón roa              ≤   0.018   ε = 0.406   α = 0.191   peso de las mal clasificadas después: 0.500   acumulado: entrenamiento 0.767  prueba 0.699



Cada tocón solo es un poco mejor que el azar (mira cómo $\varepsilon$ sube ronda a ronda: cada
uno trabaja sobre empresas más difíciles), y en cada ronda las mal clasificadas quedan pesando
exactamente la mitad. Con cinco rondas el voto ponderado todavía no supera al árbol de dos
niveles y en prueba oscila: boosting necesita más rondas para mostrar lo suyo. Con
`scikit-learn` y más tocones:


In [13]:

for rondas in [5, 20, 100]:
    ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=rondas, learning_rate=1.0, random_state=0).fit(X_train, y_train)
    evaluar(ada, f"AdaBoost, {rondas} tocones")


AdaBoost, 5 tocones                          entrenamiento 0.767   prueba 0.699
AdaBoost, 20 tocones                         entrenamiento 0.877   prueba 0.767
AdaBoost, 100 tocones                        entrenamiento 0.897   prueba 0.795



**Por qué funciona y cuándo falla.** Boosting reduce el sesgo: la suma de tocones puede
representar formas que ningún tocón alcanza solo. El riesgo simétrico: sigue bajando el error
de entrenamiento aunque la prueba ya no mejore, y es sensible a etiquetas equivocadas, a las que
da cada vez más peso.

## 5. Gradient boosting a mano: sumar escaleras que corrigen el residuo

La generalización de AdaBoost que domina los datos tabulares. La receta, para predecir un
número:

$$
F_0 = \text{promedio de } y
\qquad
r_i = y_i - F_{m-1}(x_i)
\qquad
h_m = \text{árbol chico que mejor explica } r
\qquad
F_m = F_{m-1} + \eta \cdot h_m
$$

$r_i$ es el **residuo**, lo que falta; con la pérdida cuadrática es exactamente el gradiente
negativo de la pérdida, de ahí el nombre. $\eta$ es la **tasa de aprendizaje**: con 1 se suma el
árbol completo (rápido, se sobreajusta); con 0,1 se avanza de a poco (más rondas, más suave).

Predecimos la rentabilidad (`roa`) a partir de ventas/deuda, con una sola variable para poder
dibujar la predicción: una escalera que se refina ronda a ronda.


In [14]:

x_gb = X_train[["ventas_deuda"]]; y_gb = impago.loc[X_train.index, "roa"].values
x_gb_test = X_test[["ventas_deuda"]]; y_gb_test = impago.loc[X_test.index, "roa"].values
eta = 0.5
F = np.full(len(y_gb), y_gb.mean()); F_t = np.full(len(y_gb_test), y_gb.mean())
grilla = pd.DataFrame({"ventas_deuda": np.linspace(0, 8, 400)}); F_g = np.full(len(grilla), y_gb.mean())
fig = go.Figure(); fig.add_trace(go.Scatter(x=x_gb.ventas_deuda, y=y_gb, mode="markers", name="entrenamiento", opacity=0.4))
print(f"ronda 0: F = promedio = {y_gb.mean():.4f}   ECM entrenamiento {np.mean((y_gb - F) ** 2):.5f}   ECM prueba {np.mean((y_gb_test - F_t) ** 2):.5f}")
for m in range(1, 7):
    residuo = y_gb - F
    tocon = DecisionTreeRegressor(max_depth=1, random_state=0).fit(x_gb, residuo)
    F = F + eta * tocon.predict(x_gb); F_t = F_t + eta * tocon.predict(x_gb_test); F_g = F_g + eta * tocon.predict(grilla)
    print(f"ronda {m}: corte ventas/deuda ≤ {tocon.tree_.threshold[0]:.3f}   escalón izq {tocon.tree_.value[1][0][0]:8.4f}  der {tocon.tree_.value[2][0][0]:8.4f}   "
          f"ECM entrenamiento {np.mean((y_gb - F) ** 2):.5f}   ECM prueba {np.mean((y_gb_test - F_t) ** 2):.5f}")
    if m in (1, 3, 6):
        fig.add_trace(go.Scatter(x=grilla.ventas_deuda, y=F_g, name=f"F después de {m} rondas", line=dict(width=3 if m == 6 else 1.5)))
fig.update_layout(title="Gradient boosting: la escalera se construye ronda a ronda (η = 0,5)", xaxis_title="ventas / deuda", yaxis_title="roa", xaxis_range=[0, 8])
fig.show()


ronda 0: F = promedio = 0.0637   ECM entrenamiento 0.01616   ECM prueba 0.02840
ronda 1: corte ventas/deuda ≤ 1.823   escalón izq  -0.0649  der   0.0381   ECM entrenamiento 0.01431   ECM prueba 0.02529
ronda 2: corte ventas/deuda ≤ 5.285   escalón izq  -0.0158  der   0.0890   ECM entrenamiento 0.01325   ECM prueba 0.02418
ronda 3: corte ventas/deuda ≤ 0.027   escalón izq  -0.1418  der   0.0050   ECM entrenamiento 0.01272   ECM prueba 0.02282
ronda 4: corte ventas/deuda ≤ 3.995   escalón izq  -0.0130  der   0.0382   ECM entrenamiento 0.01235   ECM prueba 0.02253
ronda 5: corte ventas/deuda ≤ 14.698   escalón izq  -0.0021  der   0.1491   ECM entrenamiento 0.01212   ECM prueba 0.02266
ronda 6: corte ventas/deuda ≤ 0.001   escalón izq  -0.1160  der   0.0016   ECM entrenamiento 0.01198   ECM prueba 0.02164



Cada ronda baja el error de entrenamiento; el de prueba baja casi siempre aquí, pero no lo
haría para siempre: por eso el número de rondas es un parámetro, y se elige con parada
temprana sobre datos de validación. Fíjate también en que el segundo corte de la ronda 3 cae en
ventas/deuda ≤ 0,027: dos empresas con ventas casi nulas y roa muy negativo, a las que el
boosting les dedica una ronda entera. Es su forma de sobreajustar.

**La familia.** Gradient boosting es la receta de arriba con cualquier pérdida derivable (para
clasificar, la logística: el residuo pasa a ser $y - p$). **XGBoost** (2014) agrega
regularización explícita en las hojas, manejo de faltantes y paralelismo; **LightGBM** (2017)
crece los árboles hoja a hoja y agrupa valores en cubetas, mucho más rápido con millones de
filas; **CatBoost** (2018) trata las categóricas sin preprocesar y usa árboles simétricos. Todos
son esta misma receta con mejoras de ingeniería. `HistGradientBoostingClassifier` de
`scikit-learn` es la versión "tipo LightGBM" que viene incluida.

**Las perillas, en orden.** Número de rondas y tasa de aprendizaje (van juntas); tamaño del
árbol (tocones para efectos simples, profundidad 3 a 6 para interacciones); submuestreo de
filas y columnas; regularización de las hojas.


In [15]:

for lr, rondas in [(1.0, 50), (0.1, 50), (0.1, 300), (0.05, 300)]:
    gb = GradientBoostingClassifier(n_estimators=rondas, learning_rate=lr, max_depth=2, random_state=0).fit(X_train, y_train)
    evaluar(gb, f"gradient boosting, η = {lr}, {rondas} rondas, prof. 2")


gradient boosting, η = 1.0, 50 rondas, prof. 2 entrenamiento 1.000   prueba 0.808
gradient boosting, η = 0.1, 50 rondas, prof. 2 entrenamiento 0.918   prueba 0.767
gradient boosting, η = 0.1, 300 rondas, prof. 2 entrenamiento 1.000   prueba 0.822
gradient boosting, η = 0.05, 300 rondas, prof. 2 entrenamiento 1.000   prueba 0.808



## 6. Donde los ensambles brillan: 30.000 clientes de tarjetas de crédito

Con 219 empresas todo empata. Cambiemos de escala: la base de tarjetas de crédito de Taiwán
(Yeh y Lien, 2009), 30.000 clientes, 23 variables (límite, edad, y para cada uno de los últimos
seis meses el atraso, la factura y el pago) y si el cliente dejó de pagar el mes siguiente
(22%). Como los clientes están desbalanceados, medimos con **AUC** (la probabilidad de que el
modelo le asigne más riesgo a un cliente que no pagó que a uno que pagó; 0,5 es azar, 1 es
perfecto), además del acierto.

Si tienes el link al archivo del curso (`taiwan_credit.csv`), pégalo en `URL_TAIWAN`; si no,
se baja de OpenML.


In [16]:

from sklearn.datasets import fetch_openml

def normalizar_link(url):
    # Convierte un link "para compartir" de Drive o Dropbox en un link de descarga directa
    import re
    m = re.search(r"drive\.google\.com/file/d/([^/]+)", url)
    if m:
        return "https://drive.google.com/uc?export=download&id=" + m.group(1)
    if "dropbox.com" in url:
        url = url.replace("www.dropbox.com", "dl.dropboxusercontent.com").replace("&dl=0", "").replace("?dl=0", "")
    return url

URL_TAIWAN = ""    # <-- pega acá el link del CSV del curso (taiwan_credit.csv) si lo tienes

if URL_TAIWAN != "":
    taiwan_df = pd.read_csv(normalizar_link(URL_TAIWAN))
    X_t = taiwan_df.drop(columns="target").apply(pd.to_numeric); y_t = taiwan_df["target"].astype(int)
    print("Taiwan leído del archivo del curso")
else:
    taiwan = fetch_openml("default-of-credit-card-clients", version=1, as_frame=True)
    X_t = taiwan.data.apply(pd.to_numeric); y_t = taiwan.target.astype(int)
    print("Taiwan bajado de OpenML")

meses = ["sep", "ago", "jul", "jun", "may", "abr"]
X_t.columns = (["limite_credito", "sexo", "educacion", "estado_civil", "edad"] + ["atraso_" + m for m in meses]
               + ["factura_" + m for m in meses] + ["pago_" + m for m in meses])
Xt_train, Xt_test, yt_train, yt_test = train_test_split(X_t, y_t, test_size=9000, random_state=0, stratify=y_t)
print("Clientes:", len(X_t), "  Variables:", X_t.shape[1], "  Fracción que no pagó:", round(y_t.mean(), 3))
print("Entrenamiento:", len(Xt_train), "  Prueba:", len(Xt_test))


Taiwan bajado de OpenML
Clientes: 30000   Variables: 23   Fracción que no pagó: 0.221
Entrenamiento: 21000   Prueba: 9000


In [17]:

modelos = {
    "árbol de profundidad 2":                 DecisionTreeClassifier(max_depth=2, random_state=0),
    "árbol de profundidad 5":                 DecisionTreeClassifier(max_depth=5, random_state=0),
    "árbol sin freno":                        DecisionTreeClassifier(random_state=0),
    "random forest, 200 árboles":             RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=0),
    "AdaBoost, 200 tocones":                  AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=200, random_state=0),
    "gradient boosting (hist), 300 rondas":   HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=4, random_state=0),
}
filas = []
for nombre, m in modelos.items():
    t0 = time.time(); m.fit(Xt_train, yt_train); seg = time.time() - t0
    p = m.predict_proba(Xt_test)[:, 1]
    filas.append((nombre, accuracy_score(yt_test, m.predict(Xt_test)), roc_auc_score(yt_test, p), seg))
resultados = pd.DataFrame(filas, columns=["modelo", "acierto prueba", "AUC prueba", "segundos"]).set_index("modelo")
print(f"Predecir siempre 'paga' acierta {1 - yt_test.mean():.3f}")
resultados.round(3)


Predecir siempre 'paga' acierta 0.779


,acierto prueba,AUC prueba,segundos
modelo,,,
árbol de profundidad 2,0.819,0.690,0.070
árbol de profundidad 5,0.819,0.746,0.158
árbol sin freno,0.718,0.604,0.574
"random forest, 200 árboles",0.817,0.775,8.286
"AdaBoost, 200 tocones",0.819,0.764,7.270
"gradient boosting (hist), 300 rondas",0.819,0.774,0.436



**Ahora sí.** Primero, el acierto no sirve para comparar: casi todos los modelos dan 82%,
que es lo que da predecir "paga" para todos, porque solo el 22% no paga. Hay que mirar el AUC.
Ahí el árbol sin freno se hunde (memoriza), el árbol de dos niveles se queda corto (0,69), el
de profundidad 5 mejora (0,75), y el random forest y el gradient boosting ganan (0,77 y
0,78) usando los 21.000 clientes para descubrir estructura que un árbol chico no alcanza. La
regla general: un ensamble gana en proporción a cuánta estructura haya que descubrir y a
cuántos datos haya para descubrirla. Con 219 empresas no había suficiente de ninguna de las dos
cosas.

Y lo que se pierde: ya no hay un reglamento que leer. Se explica con importancias y con
ejemplos, y en crédito eso tiene costo regulatorio. Es un trade-off real, no un detalle.


In [18]:

rf_t = modelos["random forest, 200 árboles"]
imp_t = pd.Series(rf_t.feature_importances_, index=X_t.columns).sort_values(ascending=False).head(10)
fig = px.bar(imp_t, orientation="h", title="Las 10 variables más importantes según el random forest (Taiwán)", labels={"value": "importancia", "index": ""})
fig.show()



## 7. Lo que hay que llevarse

- **Bagging** promedia árboles entrenados en copias sorteadas: reduce la varianza. **Random
  forest** agrega el sorteo de variables para que los árboles se parezcan menos, y regala el
  error fuera de la bolsa y la importancia de variables.
- **Boosting** entrena en serie, cada modelo concentrado en el error del anterior: reduce el
  sesgo. AdaBoost repondera empresas; gradient boosting ajusta al residuo. XGBoost, LightGBM y
  CatBoost son gradient boosting con mejoras de ingeniería.
- Con pocos datos y pocas variables, un árbol chico empata con todo. Con muchos datos, los
  ensambles ganan y el árbol sin freno se hunde.
- Qué preguntar cuando alguien presenta un ensamble: cuántas rondas o árboles y cómo se eligió
  el número, qué error da en datos que nunca vio, qué variables pesan y si tienen sentido de
  negocio, y cómo se explica una predicción individual.

## Ejercicios

**Ejercicio 1.** En la sección 2, repite el bagging a mano con `B = 25` en vez de 7. ¿Sube el
acierto de la votación en prueba? ¿Y el fuera de la bolsa? ¿Cuántas empresas quedan sin
ningún árbol que no las haya visto?

**Ejercicio 2.** En Taiwán, entrena el gradient boosting con `max_iter` de 30, 100, 300 y
1.000 (con `learning_rate=0.05`). ¿Mejora siempre el AUC de prueba? ¿Qué le dirías a alguien
que propone "más rondas"?

**Ejercicio 3.** Entrena el random forest de Taiwán con `max_features` igual a 1, 4 (cerca de
la raíz cuadrada de 23) y 23 (todas las variables, que es bagging puro). ¿Cuál da mejor AUC?
Relaciónalo con $\rho$ en la fórmula de la varianza del promedio.



## Soluciones


In [19]:

# SOLUCIÓN 1 -- más árboles en el bagging a mano
rng = np.random.default_rng(11); B25 = 25
muestras25 = [rng.integers(0, n, n) for _ in range(B25)]
arboles25 = [DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train.iloc[m], y_train.iloc[m]) for m in muestras25]
votos25 = np.column_stack([a.predict(X_test) for a in arboles25]); mayoria25 = (votos25.mean(axis=1) > 0.5).astype(int)
cuentas25 = np.column_stack([np.bincount(m, minlength=n) for m in muestras25])
oob = []; sin_arbol = 0
for j in range(n):
    fuera = [b for b in range(B25) if cuentas25[j, b] == 0]
    if not fuera: sin_arbol += 1; continue
    oob.append(int(np.mean([arboles25[b].predict(X_train.iloc[[j]])[0] for b in fuera]) > 0.5) == y_train.iloc[j])
print(f"B = 25: votación en prueba {accuracy_score(y_test, mayoria25):.3f}   fuera de la bolsa {np.mean(oob):.3f}   empresas sin ningún árbol que no las viera: {sin_arbol}")
print("Con 25 árboles, cada empresa queda fuera de unos 9: el OOB se estabiliza. La votación en prueba cambia poco: el límite es la información de los datos, no el número de árboles.")


B = 25: votación en prueba 0.753   fuera de la bolsa 0.740   empresas sin ningún árbol que no las viera: 0
Con 25 árboles, cada empresa queda fuera de unos 9: el OOB se estabiliza. La votación en prueba cambia poco: el límite es la información de los datos, no el número de árboles.


In [20]:

# SOLUCIÓN 2 -- cuántas rondas
for it in [30, 100, 300, 1000]:
    t0 = time.time()
    m = HistGradientBoostingClassifier(max_iter=it, learning_rate=0.05, max_depth=4, early_stopping=False, random_state=0).fit(Xt_train, yt_train)
    print(f"{it:5d} rondas: AUC prueba {roc_auc_score(yt_test, m.predict_proba(Xt_test)[:, 1]):.4f}   ({time.time() - t0:.0f} s)")
print("El AUC sube, se estanca y termina bajando: más rondas es más sobreajuste. Se elige con parada temprana sobre validación, no 'lo más que se pueda'.")


   30 rondas: AUC prueba 0.7662   (0 s)
  100 rondas: AUC prueba 0.7720   (1 s)
  300 rondas: AUC prueba 0.7735   (1 s)
 1000 rondas: AUC prueba 0.7670   (3 s)
El AUC sube, se estanca y termina bajando: más rondas es más sobreajuste. Se elige con parada temprana sobre validación, no 'lo más que se pueda'.


In [21]:

# SOLUCIÓN 3 -- cuántas variables por corte
for mf in [1, 4, 23]:
    m = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, max_features=mf, n_jobs=-1, random_state=0).fit(Xt_train, yt_train)
    print(f"max_features = {mf:2d}: AUC prueba {roc_auc_score(yt_test, m.predict_proba(Xt_test)[:, 1]):.4f}")
print("Con todas las variables (bagging puro) los árboles se parecen (rho alto) y el promedio ayuda menos: es el peor de los tres. Con 1 o con 4 los árboles se parecen menos y el bosque mejora; aquí las dos opciones empatan.")


max_features =  1: AUC prueba 0.7761
max_features =  4: AUC prueba 0.7747
max_features = 23: AUC prueba 0.7657
Con todas las variables (bagging puro) los árboles se parecen (rho alto) y el promedio ayuda menos: es el peor de los tres. Con 1 o con 4 los árboles se parecen menos y el bosque mejora; aquí las dos opciones empatan.
